In [5]:
# !pip install -U langchain
# !pip install langchain_community
# !pip install -qU pymupdf

In [6]:
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_community.vectorstores import FAISS

FILE_PATH = "./data/virtual_tech_internal_handbook.pdf"

loader = PyMuPDFLoader(FILE_PATH)
docs = loader.load()
len(docs)

llm = ChatOpenAI(model="gpt-5", temperature=0)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    length_function=len,
    is_separator_regex=False,
)

split_docs = text_splitter.split_documents(docs)


C:\Users\user\AppData\Local\Temp\ipykernel_21248\3135080492.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


In [8]:
print(len(docs))
print(docs[0].page_content[:300])

9
VIRTUALTECH · INTERNAL HANDBOOK
가상Tech
업무 가이드
함께 일하는 방식부터 보안, 개발, 장애 대응, 복지까지
구성원이 매일 참고하는 실무 기준
문서 버전
시행일
문서 등급
문의
v1.0
2026-09-01
사내 공개
People Operations
이 문서의 목적
가상Tech 구성원이 반복해서 찾는 사내 정책과 표준 절차를 한 문서에 정리했습니다.
실제 회사가 아닌 RAG 검색·질의응답 실습용 가상 문서입니다.


In [9]:
def show_metadata(docs):
    if docs:
        print(["[metadata]"])
        print(list(docs[0].metadata.keys()))
        print("\n[examples]")
        max_key_length = max(len(k) for k in docs[0].metadata.keys())
        for k, v in docs[0].metadata.items():
            print(f"{k:<{max_key_length}} : {v}")

show_metadata(docs)

['[metadata]']
['producer', 'creator', 'creationdate', 'source', 'file_path', 'total_pages', 'format', 'title', 'author', 'subject', 'keywords', 'moddate', 'trapped', 'modDate', 'creationDate', 'page']

[examples]
producer     : ReportLab PDF Library - (opensource)
creator      : (unspecified)
creationdate : 2026-09-26T11:41:50+09:00
source       : ./data/virtual_tech_internal_handbook.pdf
file_path    : ./data/virtual_tech_internal_handbook.pdf
total_pages  : 9
format       : PDF 1.4
title        : 가상Tech 업무 가이드
author       : 가상Tech People Operations
subject      : RAG 실습용 가상 사내 업무 가이드
keywords     : 
moddate      : 2026-09-26T11:41:50+09:00
trapped      : 
modDate      : D:20260926114150+09'00'
creationDate : D:20260926114150+09'00'
page         : 0


In [10]:
# `pretty_print_docs` 함수는 문서 리스트를 예쁘게 출력하는 헬퍼 함수입니다.
def pretty_print_docs(docs):
    print(
        f"\n{'-'*100}\n".join(
            [f"문서 {i+1}:\n\n" + d.page_content for i, d in enumerate(docs)]
        )
    )

In [ ]:
embeddings = OpenAIEmbeddings()

db = FAISS.from_documents(split_docs, embeddings)

retriever = db.as_retriever(search_kwargs={"k":4})

In [12]:
retrieverd = retriever.invoke("연차를 사용하려면 최소 며칠 전에 신청해야 하나요?")

pretty_print_docs(retrieverd)

문서 1:

• 국내 출장 교통·숙박은 출발 5영업일 전 신청합니다. 숙박 한도는 1박 15만원(세금 포함)입니다.
• 택시는 23:00 이후 퇴근, 대중교통 미운행, 고객 일정 등 업무상 필요가 있을 때 실비 지원합니다.
7.3 주요 복지
제도
지원 내용
신청
성장 지원
연 120만원 교육·도서·컨퍼런스
LearnHub에서 사전 승인
건강 지원
연 60만원 운동·상담
월별 영수증 제출
장비 지원
3년 주기 노트북 교체, 모니터 1대
#help-it 티켓
리프레시
근속 3년마다 유급 10영업일
3개월 전 팀과 일정 협의
휴가 중 연락
휴가자는 업무 메시지에 응답할 의무가 없습니다. 인수인계 문서에 대체 담당자,
진행 상태, 긴급 연락 기준을 남기고 Slack 상태와 캘린더를 갱신합니다.
----------------------------------------------------------------------------------------------------
문서 2:

가상Tech 업무 가이드
사내용 · v1.0
문서 책임: People Operations
08
CHAPTER 07
휴가·비용·복지
예측 가능한 신청 절차로 개인의 휴식과 팀 운영을 함께 지킵니다.
7.1 휴가
구분
신청 시점
승인·증빙
연차
1일 전
팀장 승인, 3일 이상은 2주 전 협의
반차
당일 10:00 전
팀장 승인
병가
가능한 즉시
3일 초과 시 진료 확인서 제출
경조휴가
사유 확인 즉시
People Ops 확인, 사내 규정 일수 적용
7.2 비용 처리
• 법인카드 사용 내역은 영수증과 목적을 지출일로부터 5영업일 안에 ExpenseHub에 등록합니다.
• 건당 10만원 이상 소프트웨어·비품은 구매 전에 팀장 승인을 받습니다. 연간 구독은 IT 자산 검토도 필요합니다.
• 국내 출장 교통·숙박은 출발 5영업일 전 신청합니다. 숙박 한도는 1박 15만원(세금 포함)입니다.
-------------------------------------------------------------

In [13]:
for i, d in enumerate(retrieverd, 1):
    print(i, "연차" in d.page_content,
          "1일 전" in d.page_content,
          d.metadata.get("page"))

1 False False 7
2 True True 7
3 False False 2
4 False False 1


In [14]:
def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

prompt = PromptTemplate.from_template(
    """아래 [문서]만 근거로 질문에 한국어로 답하세요.
    [문서]에 답이 없으면 "문서에서 찾을 수 없습니다"라고 답하세요.
    
    [문서]
    {context}
    
    [질문]
    {question}
    
    [답변]"""
)

llm = ChatOpenAI(model="gpt-5", temperature=0)

rag_chain = (
    {"context" : retriever | format_docs,
     "question" : RunnablePassthrough()}
     | prompt
     | llm
     | StrOutputParser()
)

In [15]:
print(rag_chain.invoke("연차를 사용하려면 최소 며칠 전에 신청해야 하나요?"))

연차는 최소 1일 전에 신청해야 합니다. 단, 3일 이상 사용하는 경우에는 2주 전 사전 협의가 필요합니다.


In [16]:
print(rag_chain.invoke("서울 출장 시 숙박비는 최대 얼마까지 지원되나요?"))

1박당 최대 15만원(세금 포함)입니다.


In [17]:
print(rag_chain.invoke("출장을 다녀온 후 비용 정산은 언제까지 완료해야 하나요?"))

지출일(각 비용 발생일)로부터 5영업일 이내에 영수증과 목적을 첨부해 ExpenseHub에 등록해야 합니다.


In [18]:
print(rag_chain.invoke("회사 노트북을 분실했을 경우 어떤 절차로 신고해야 하나요?"))

문서에서 찾을 수 없습니다


In [19]:
print(rag_chain.invoke("재택근무를 신청하려면 어떤 조건을 충족해야 하나요?"))

- 기본 조건: 재택은 주 2회까지 가능하며 주 단위로 관리되어 이월되지 않습니다.
- 신청 방법: 해당 주의 재택 예정일을 매주 월요일까지 캘린더에 ‘WFH’로 표시합니다.
- 예외: 고객 방문·건강상 사유 등 규정 외 필요 시 팀장과 People Ops 승인 필요.


In [20]:
print(rag_chain.invoke("직원이 업무 관련 교육을 받을 경우 회사에서 지원하는 비용 기준은 어떻게 되나요?"))

연 120만원 한도 내에서 교육·도서·컨퍼런스 비용을 지원하며, LearnHub에서 사전 승인을 받아야 합니다.


In [21]:
print(rag_chain.invoke("출장 중 택시비를 회사 경비로 처리할 수 있는 경우는 언제인가요?"))

다음과 같이 업무상 필요가 있을 때 택시비를 실비로 회사 경비 처리할 수 있습니다.
- 23:00 이후 퇴근 시
- 대중교통이 미운행일 때
- 고객 일정 등 업무상 필요가 있을 때


In [22]:
print(rag_chain.invoke("개인 사정으로 출근 시간이 늦어질 경우 어떤 방식으로 보고해야 하나요?"))

문서에서 찾을 수 없습니다


In [23]:
print(rag_chain.invoke("업무 중 시스템 장애가 발생했을 때 가장 먼저 해야 하는 조치는 무엇인가요?"))

장애를 감지한 사람이 즉시 Slack의 #help-urgent에 ‘증상, 시작 시각, 영향, 관측 링크’를 게시하고 PagerDuty를 호출합니다.


In [24]:
print(rag_chain.invoke("제주도로 2박 3일 출장을 가는 경우 숙박비 지원 한도와 출장비 정산 기한을 각각 알려주세요."))

- 숙박비 지원 한도: 1박 15만원(세금 포함) → 2박 총 30만원
- 출장비 정산 기한: 지출일로부터 5영업일 이내(ExpenseHub 등록)


In [27]:
with open("question.text", encoding="utf-8") as f:
    lines = f.read().splitlines()

question = [l.split(". ", 1)[1] for l in lines if l[:1].isdigit()]

for i, q in enumerate(question, 1):
    print(f"Q{i}. {q}")
    print("A.", rag_chain.invoke(q))
    print("-"*60)

Q1. 연차를 사용하려면 최소 며칠 전에 신청해야 하나요?
A. 최소 1일 전에 신청해야 합니다. (단, 3일 이상 연차는 2주 전 사전 협의)
------------------------------------------------------------
Q2. 서울 출장 시 숙박비는 최대 얼마까지 지원되나요?
A. 1박당 최대 15만원(세금 포함)까지 지원됩니다.
------------------------------------------------------------
Q3. 출장을 다녀온 후 비용 정산은 언제까지 완료해야 하나요?
A. 지출 건별로 영수증과 사용 목적을 첨부해 지출일로부터 5영업일 이내에 ExpenseHub에 등록해야 합니다.
------------------------------------------------------------
Q4. 회사 노트북을 분실했을 경우 어떤 절차로 신고해야 하나요?
A. 문서에서 찾을 수 없습니다
------------------------------------------------------------
Q5. 재택근무를 신청하려면 어떤 조건을 충족해야 하나요?
A. - 주 2회 범위 내에서만 가능하며, 기준은 주 단위입니다(이월 불가).
- 매주 월요일까지 개인 캘린더에 ‘WFH’로 표시해야 합니다.
- 고객 방문·건강상 사유 등 예외가 필요한 경우 팀장과 People Ops의 승인이 필요합니다.
------------------------------------------------------------
Q6. 직원이 업무 관련 교육을 받을 경우 회사에서 지원하는 비용 기준은 어떻게 되나요?
A. - 연 120만원 한도 내에서 교육·도서·컨퍼런스 비용을 지원합니다.
- 신청은 LearnHub에서 사전 승인을 받아야 합니다.
------------------------------------------------------------
Q7. 출장 중 택시비를 회사 경비로 처리할 수 있는 경우는 언제인가요?
A